# Lesson 000.07 -- The First Up-Lift: Context Transition MSS-0 to MSS-1

AuthorityEffect: None.

> **Claim-typing legend**
> - **[Pinned]**: reproduced from verified checked-in Grammar 000 and Up/Down map specifications (`updown_maps.py`).
> - **[Executed]**: verified by deterministic Python execution in this notebook.
> - **[Declared]**: pedagogical convention or beginner framing.
> - **[Open]**: questions left for future exploration.

## 1. The Big Idea: Context Lift $u_0$ and Forgetful Projection $d_1$

[Pinned] In the tower architecture, moving from Level 0 to Level 1 is governed by an explicit context-map pair $(u_0, d_1)$:
- **The Up-Lift ($u_0: \text{MSS-0} \to \text{MSS-1}$):**
  At MSS-0, the ONLY legal packet is the bare distinction `Diff()` (`<>`). The lift preserves `Diff` identically:
  $$u_0(\text{Diff}) = \text{Diff}.$$
  Calling $u_0$ on any packet not active at Level 0 (such as `<0>`) is an explicit precondition violation and raises `ValueError`.
- **The Down Map ($d_1: \text{MSS-1} \to \text{MSS-0} \mid \text{NonDescentReceipt}$):**
  The forgetful map retracts terms from MSS-1 back to MSS-0 when restricted to the image of $u_0$:
  $$d_1(u_0(\text{Diff})) = \text{Diff}.$$
  Identity on old structure is preserved.

[Declared] **Glyphs vs. Numbers:**
The symbols `0` and `1` introduced at MSS-1 are initially uninterpreted wire glyphs in the active binary alphabet $\{0, 1\}$. They become numerical integers only after an explicit semantic decoding interpretation (e.g. `int(atom.word, 2)`) is declared.

## 2. Non-Descent Refusal: New Structure Cannot Silently Descend

[Pinned] A critical integrity invariant of the tower is:
> *"New structure cannot silently descend as old structure."*

If the down map $d_1$ simply returned `Diff()` whenever it encountered an Atom `<0>` or an application `op<...>`, a caller could not tell whether the term genuinely was the base distinction or whether new structure was quietly coerced to look like nothing.

To prevent this silent corruption:
- On genuinely new MSS-1 structure, $d_1$ returns an explicit, typed **`NonDescentReceipt(original=packet)`**.
- A refusal is made visible, typed, and auditable.


In [ ]:
# [Pinned] Environment Preflight & Project Root Resolution
# Pinned revision: 3f3399766108adc4c4e7615dce2573a57be57dd6
# Supported environment: Python 3.12+ (e.g. py -3.12 or pixi run)
# Project install: pip install -e .
import sys
from pathlib import Path

ROOT_DIR = Path.cwd().resolve()
found = False
while ROOT_DIR.parent != ROOT_DIR:
    if (
        (ROOT_DIR / "pfp").exists()
        or (ROOT_DIR / ".git").exists()
        or (ROOT_DIR / "pixi.toml").exists()
    ):
        if (ROOT_DIR / "pfp").exists():
            found = True
            break
    ROOT_DIR = ROOT_DIR.parent

if found and str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))
elif not found:
    raise RuntimeError(
        "Could not locate 'pfp' root. "
        "Ensure you execute from a repository checkout or install via: pip install -e ."
    )

from pfp.domain.tower import grammar, updown_maps

diff = grammar.Diff()
atom0 = grammar.Atom("0")
atom1 = grammar.Atom("1")

# 1. Identity on old structure
lifted = updown_maps.u_0(diff)
assert lifted == diff
assert updown_maps.d_1(lifted) == diff
print("Verified u_0(Diff) == Diff and d_1(u_0(Diff)) == Diff")

# 2. Non-descent on genuinely new MSS-1 structure
receipt0 = updown_maps.d_1(atom0)
assert isinstance(receipt0, updown_maps.NonDescentReceipt)
assert receipt0.original == atom0

receipt1 = updown_maps.d_1(atom1)
assert isinstance(receipt1, updown_maps.NonDescentReceipt)
assert receipt1.original == atom1
print("Verified typed NonDescentReceipt on Atom('0') and Atom('1')")

# 3. Precondition violation on u_0
try:
    updown_maps.u_0(atom0)
    assert False, "Expected ValueError on u_0(atom0)"
except ValueError as e:
    assert "level-0-active" in str(e).lower()
    print("Verified u_0 raises ValueError on non-level-0 packet:", e)

## 3. Diagnostic Probe (2026 Micro-Attention Check)

**Question:** What does $d_1(\text{Atom("0")})$ return?

- **A:** `Diff()`, coercing the new atom to the base distinction.
- **B:** `None`.
- **C:** `NonDescentReceipt(original=Atom("0"))`, making the refusal explicit and typed.

*(Correct answer: **C**. New MSS-1 structure cannot silently coerce to old MSS-0 structure; it produces a typed `NonDescentReceipt`.)*

In [ ]:
# Interactive Diagnostic Probe Check
student_choice = "C"  # Change to experiment
assert student_choice == "C", "Incorrect! Expected C: d_1 returns typed NonDescentReceipt"

# Mechanically verify the property
ret = updown_maps.d_1(grammar.Atom("0"))
assert isinstance(ret, updown_maps.NonDescentReceipt)
assert ret.original == grammar.Atom("0")
assert not isinstance(ret, grammar.Diff)

print("Diagnostic Probe: PASSED (NonDescentReceipt emitted)")

## 4. Student Exercises (Ages 5 to 500)

1. **Exercise 1 (Age 5):** If you lift a plain box into Level 1, what do you have? *(Answer: The same plain box!)*
2. **Exercise 2 (Age 15):** In Python, call $d_1(u_0(\text{Diff}()))$. What is the result? Test it below!
3. **Exercise 3 (Age 500):** Try calling $u_0(\text{Atom("0")})$. What exception is raised and why? *(Answer: ValueError, because `<0>` is not active at Level 0, violating the precondition of $u_0$.)*

In [ ]:
# Exercise Solution & Verification
# 1. Round-trip on Diff
diff_term = grammar.Diff()
assert updown_maps.d_1(updown_maps.u_0(diff_term)) == diff_term

# 2. Descent on an application holding an atom
app_term = grammar.App((grammar.Atom("0"),))
app_receipt = updown_maps.d_1(app_term)
assert isinstance(app_receipt, updown_maps.NonDescentReceipt)
assert app_receipt.original == app_term

print("Student exercises verified successfully: round-trip and non-descent receipts.")

## 5. Anchors & References

| Anchor Type | Target Link |
| :--- | :--- |
| **Curriculum Anchor** | [Up/Down Maps Specification](../../../../symbol_game/domain/tower/updown_maps.py) |
| **Authority Effect** | `None` |
| **Invariant** | Identity preservation on old structure; typed receipt on non-descent |